# Autômatos — Assistência Técnica de Impressoras

Projeto desenvolvido por:

Felipe Ferreira - 2993620

Nicole Vieira - 2438577

Sthefany Alves - 2457820

Vitória Santos - 2392314

Igor Xavier - 2352643

In [16]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
import re
from dataclasses import dataclass
import ipywidgets as widgets
from IPython.display import display, clear_output
print("Ambiente preparado.")

Ambiente preparado.


## R3 — AFD, desenho, delta, tabela e testes AFD × RegEx
Os AFDs abaixo reconhecem códigos de chamados `CH` + quatro dígitos e impressoras `IMP-` + três dígitos. Transições não listadas vão para o estado morto.

In [17]:
class AFD:
    def __init__(self, nome, estados, alfabeto, inicial, finais, transicoes):
        self.nome, self.estados, self.alfabeto = nome, list(estados), set(alfabeto)
        self.inicial, self.finais, self.transicoes = inicial, set(finais), dict(transicoes)
        self.morto = "MORTO"
    def delta(self, estado, simbolo):
        if estado == self.morto: return self.morto
        return self.transicoes.get((estado, simbolo), self.morto)
    def aceita(self, palavra):
        estado = self.inicial
        for c in palavra: estado = self.delta(estado, c)
        return estado in self.finais
    def desenhar(self):
        print(f"Desenho textual do AFD: {self.nome}")
        print(f"Entrada -> {self.inicial}")
        for (origem, simbolo), destino in self.transicoes.items():
            print(f"({origem}) --{simbolo}--> ({destino})")
        print("Estado(s) final(is):", ", ".join(sorted(self.finais)))
        print("Transições não listadas vão para MORTO.")
    def tabela(self):
        simbolos = sorted(self.alfabeto)
        print(f"Tabela de transições — {self.nome}")
        print("Estado\\Símbolo | " + " | ".join(simbolos))
        for q in self.estados + [self.morto]:
            print(f"{q:<14} | " + " | ".join(self.delta(q, c) for c in simbolos))

def afd_chamado():
    t = {("q0","C"):"q1", ("q1","H"):"q2"}
    for d in "0123456789":
        t.update({("q2",d):"q3", ("q3",d):"q4", ("q4",d):"q5", ("q5",d):"q6"})
    return AFD("Chamado CHdddd", ["q0","q1","q2","q3","q4","q5","q6"],
               "CH0123456789", "q0", {"q6"}, t)

def afd_impressora():
    t = {("q0","I"):"q1", ("q1","M"):"q2", ("q2","P"):"q3", ("q3","-"):"q4"}
    for d in "0123456789":
        t.update({("q4",d):"q5", ("q5",d):"q6", ("q6",d):"q7"})
    return AFD("Impressora IMP-ddd", ["q0","q1","q2","q3","q4","q5","q6","q7"],
               "IMP-0123456789", "q0", {"q7"}, t)

AFDS = {
 "Chamado": {"afd":afd_chamado(), "regex":r"CH[0-9]{4}",
   "yes":["CH1234","CH0001","CH9876","CH4321","CH9999"],
   "no":["","CH123","CH12345","1234","CH12AB","ch1234"]},
 "Impressora": {"afd":afd_impressora(), "regex":r"IMP-[0-9]{3}",
   "yes":["IMP-123","IMP-001","IMP-999","IMP-456","IMP-007"],
   "no":["","IMP12","IMP-12","IMP-1234","IMP-ABC","imp-123"]}
}
for cfg in AFDS.values():
    cfg["afd"].desenhar()
    cfg["afd"].tabela()

Desenho textual do AFD: Chamado CHdddd
Entrada -> q0
(q0) --C--> (q1)
(q1) --H--> (q2)
(q2) --0--> (q3)
(q3) --0--> (q4)
(q4) --0--> (q5)
(q5) --0--> (q6)
(q2) --1--> (q3)
(q3) --1--> (q4)
(q4) --1--> (q5)
(q5) --1--> (q6)
(q2) --2--> (q3)
(q3) --2--> (q4)
(q4) --2--> (q5)
(q5) --2--> (q6)
(q2) --3--> (q3)
(q3) --3--> (q4)
(q4) --3--> (q5)
(q5) --3--> (q6)
(q2) --4--> (q3)
(q3) --4--> (q4)
(q4) --4--> (q5)
(q5) --4--> (q6)
(q2) --5--> (q3)
(q3) --5--> (q4)
(q4) --5--> (q5)
(q5) --5--> (q6)
(q2) --6--> (q3)
(q3) --6--> (q4)
(q4) --6--> (q5)
(q5) --6--> (q6)
(q2) --7--> (q3)
(q3) --7--> (q4)
(q4) --7--> (q5)
(q5) --7--> (q6)
(q2) --8--> (q3)
(q3) --8--> (q4)
(q4) --8--> (q5)
(q5) --8--> (q6)
(q2) --9--> (q3)
(q3) --9--> (q4)
(q4) --9--> (q5)
(q5) --9--> (q6)
Estado(s) final(is): q6
Transições não listadas vão para MORTO.
Tabela de transições — Chamado CHdddd
Estado\Símbolo | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | C | H
q0             | MORTO | MORTO | MORTO | MORTO | MORTO | MORTO | MOR

In [18]:
def testar(nome):
    cfg = AFDS[nome]; afd, padrao = cfg["afd"], cfg["regex"]
    casos = [(x,True) for x in cfg["yes"]] + [(x,False) for x in cfg["no"]]
    divergencias = []
    print(f"\n{nome}: RegEx = ^(?:{padrao})$")
    print(f"{'Entrada':<14} {'Esperado':<9} {'AFD':<7} {'RegEx':<7} Resultado")
    for x, esperado in casos:
        a = afd.aceita(x)
        r = re.fullmatch(padrao, x) is not None
        ok = a == r == esperado
        print(f"{(x or '∅ vazia'):<14} {str(esperado):<9} {str(a):<7} {str(r):<7} {'OK' if ok else 'DIVERGÊNCIA'}")
        if not ok: divergencias.append((x, esperado, a, r))
    print(f"Total: {len(casos)} | Divergências: {len(divergencias)}")
    return divergencias

todas = []
for nome in AFDS: todas.extend((nome,d) for d in testar(nome))
assert not todas, f"Divergências encontradas: {todas}"
print("RESULTADO GERAL: 0 DIVERGÊNCIAS")


Chamado: RegEx = ^(?:CH[0-9]{4})$
Entrada        Esperado  AFD     RegEx   Resultado
CH1234         True      True    True    OK
CH0001         True      True    True    OK
CH9876         True      True    True    OK
CH4321         True      True    True    OK
CH9999         True      True    True    OK
∅ vazia        False     False   False   OK
CH123          False     False   False   OK
CH12345        False     False   False   OK
1234           False     False   False   OK
CH12AB         False     False   False   OK
ch1234         False     False   False   OK
Total: 11 | Divergências: 0

Impressora: RegEx = ^(?:IMP-[0-9]{3})$
Entrada        Esperado  AFD     RegEx   Resultado
IMP-123        True      True    True    OK
IMP-001        True      True    True    OK
IMP-999        True      True    True    OK
IMP-456        True      True    True    OK
IMP-007        True      True    True    OK
∅ vazia        False     False   False   OK
IMP12          False     False   False   OK
IMP

## R4 — Máquina de estados do chamado
Inclui seis estados, mais de seis transições, recusa de eventos inválidos com lista de eventos permitidos e quatro fluxos de teste.

In [19]:
ESTADOS = {"ABERTO","EM_ATENDIMENTO","EM_MANUTENCAO","AGUARDANDO_PECA","RESOLVIDO","FECHADO"}
TRANSICOES = {
 ("ABERTO","INICIAR"):"EM_ATENDIMENTO",
 ("EM_ATENDIMENTO","ENCAMINHAR_MANUTENCAO"):"EM_MANUTENCAO",
 ("EM_ATENDIMENTO","RESOLVER"):"RESOLVIDO",
 ("EM_MANUTENCAO","AGUARDAR_PECA"):"AGUARDANDO_PECA",
 ("AGUARDANDO_PECA","PECA_RECEBIDA"):"EM_MANUTENCAO",
 ("EM_MANUTENCAO","RESOLVER"):"RESOLVIDO",
 ("RESOLVIDO","FECHAR"):"FECHADO",
 ("RESOLVIDO","REABRIR"):"EM_ATENDIMENTO"
}
def permitidos(estado):
    return sorted(e for (q,e) in TRANSICOES if q == estado)
def transitar(estado, evento):
    if (estado,evento) not in TRANSICOES:
        return estado, False, f"Recusado. Eventos permitidos em {estado}: {', '.join(permitidos(estado)) or 'nenhum'}"
    destino = TRANSICOES[(estado,evento)]
    return destino, True, f"{estado} --{evento}--> {destino}"

fluxos = [
 ("Válido 1",["INICIAR","RESOLVER","FECHAR"],True),
 ("Válido 2",["INICIAR","ENCAMINHAR_MANUTENCAO","AGUARDAR_PECA","PECA_RECEBIDA","RESOLVER","FECHAR"],True),
 ("Inválido 1",["FECHAR"],False),
 ("Inválido 2",["INICIAR","AGUARDAR_PECA"],False)
]
for nome, eventos, esperado in fluxos:
    estado="ABERTO"; recusado=False; hist=[estado]
    print("\\n"+nome)
    for ev in eventos:
        novo, ok, msg = transitar(estado,ev); print(msg)
        if not ok: recusado=True; break
        estado=novo; hist.append(estado)
    valido = not recusado and estado=="FECHADO"
    print("Histórico:", " -> ".join(hist))
    assert valido == esperado

\nVálido 1
ABERTO --INICIAR--> EM_ATENDIMENTO
EM_ATENDIMENTO --RESOLVER--> RESOLVIDO
RESOLVIDO --FECHAR--> FECHADO
Histórico: ABERTO -> EM_ATENDIMENTO -> RESOLVIDO -> FECHADO
\nVálido 2
ABERTO --INICIAR--> EM_ATENDIMENTO
EM_ATENDIMENTO --ENCAMINHAR_MANUTENCAO--> EM_MANUTENCAO
EM_MANUTENCAO --AGUARDAR_PECA--> AGUARDANDO_PECA
AGUARDANDO_PECA --PECA_RECEBIDA--> EM_MANUTENCAO
EM_MANUTENCAO --RESOLVER--> RESOLVIDO
RESOLVIDO --FECHAR--> FECHADO
Histórico: ABERTO -> EM_ATENDIMENTO -> EM_MANUTENCAO -> AGUARDANDO_PECA -> EM_MANUTENCAO -> RESOLVIDO -> FECHADO
\nInválido 1
Recusado. Eventos permitidos em ABERTO: INICIAR
Histórico: ABERTO
\nInválido 2
ABERTO --INICIAR--> EM_ATENDIMENTO
Recusado. Eventos permitidos em EM_ATENDIMENTO: ENCAMINHAR_MANUTENCAO, RESOLVER
Histórico: ABERTO -> EM_ATENDIMENTO


## R5 — Mini-scanner
Reconhece mais de oito tipos de tokens, palavras reservadas, inteiros/reais, operadores, códigos, comentários e espaços. Usa maximal munch (`==` antes de `=`; `<=` antes de `<`). Erros mostram linha, coluna e dica, sem interromper a análise.

In [20]:
@dataclass
class Token:
    tipo:str; lexema:str; linha:int; coluna:int
@dataclass
class Erro:
    char:str; linha:int; coluna:int; dica:str

RESERVADAS={"ABRIR","CONSULTAR","INICIAR","MANUTENCAO","RESOLVER","FECHAR","PRIORIDADE","TEMPO"}
PADROES=[
 ("CHAMADO",re.compile(r"CH[0-9]{4}")), ("IMPRESSORA",re.compile(r"IMP-[0-9]{3}")),
 ("REAL",re.compile(r"[0-9]+\.[0-9]+")), ("INTEIRO",re.compile(r"[0-9]+")),
 ("IDENTIFICADOR",re.compile(r"[A-Za-z_][A-Za-z0-9_]*")),
 ("IGUAL_IGUAL",re.compile(r"==")), ("MENOR_IGUAL",re.compile(r"<=")),
 ("MAIOR_IGUAL",re.compile(r">=")), ("DIFERENTE",re.compile(r"!=")),
 ("IGUAL",re.compile(r"=")), ("MENOR",re.compile(r"<")), ("MAIOR",re.compile(r">")),
 ("MENOS",re.compile(r"-")), ("MAIS",re.compile(r"\+")), ("MULTIPLICACAO",re.compile(r"\*")),
 ("DIVISAO",re.compile(r"/")), ("ABRE_PARENTESE",re.compile(r"\(")),
 ("FECHA_PARENTESE",re.compile(r"\)")), ("VIRGULA",re.compile(r",")),
 ("PONTO_E_VIRGULA",re.compile(r";"))
]
def scanner(codigo):
    ts, erros=[] , []
    i=0; linha=1; coluna=1
    while i<len(codigo):
        c=codigo[i]
        if c in " \t\r": i+=1; coluna+=1; continue
        if c=="\n": i+=1; linha+=1; coluna=1; continue
        if c=="#":
            while i<len(codigo) and codigo[i]!="\n": i+=1; coluna+=1
            continue
        candidatos=[]
        for tipo,pat in PADROES:
            m=pat.match(codigo,i)
            if m: candidatos.append((len(m.group()),tipo,m.group()))
        if candidatos:
            n,tipo,lex=max(candidatos,key=lambda x:x[0])
            if tipo=="IDENTIFICADOR" and lex in RESERVADAS: tipo="RESERVADA"
            ts.append(Token(tipo,lex,linha,coluna)); i+=n; coluna+=n
        else:
            dica = "Use comandos como ABRIR CH1234 IMP-001." if c=="@" else "Revise o comando e os códigos CH1234/IMP-001."
            erros.append(Erro(c,linha,coluna,dica)); i+=1; coluna+=1
    return ts,erros

demo="PRIORIDADE CH1234 == 2 <= 3 @\nTEMPO CH1234 = 1.5\nCONSULTAR CH1234 $"
ts,es=scanner(demo)
for t in ts: print(t)
print("Erros:")
for e in es: print(f"Linha {e.linha}, coluna {e.coluna}: {e.char!r}. Dica: {e.dica}")
assert any(t.tipo=="IGUAL_IGUAL" for t in ts) and any(t.tipo=="MENOR_IGUAL" for t in ts)
assert any(t.lexema=="CONSULTAR" for t in ts) and len(es)==2

Token(tipo='RESERVADA', lexema='PRIORIDADE', linha=1, coluna=1)
Token(tipo='CHAMADO', lexema='CH1234', linha=1, coluna=12)
Token(tipo='IGUAL_IGUAL', lexema='==', linha=1, coluna=19)
Token(tipo='INTEIRO', lexema='2', linha=1, coluna=22)
Token(tipo='MENOR_IGUAL', lexema='<=', linha=1, coluna=24)
Token(tipo='INTEIRO', lexema='3', linha=1, coluna=27)
Token(tipo='RESERVADA', lexema='TEMPO', linha=2, coluna=1)
Token(tipo='CHAMADO', lexema='CH1234', linha=2, coluna=7)
Token(tipo='IGUAL', lexema='=', linha=2, coluna=14)
Token(tipo='REAL', lexema='1.5', linha=2, coluna=16)
Token(tipo='RESERVADA', lexema='CONSULTAR', linha=3, coluna=1)
Token(tipo='CHAMADO', lexema='CH1234', linha=3, coluna=11)
Erros:
Linha 1, coluna 29: '@'. Dica: Use comandos como ABRIR CH1234 IMP-001.
Linha 3, coluna 18: '$'. Dica: Revise o comando e os códigos CH1234/IMP-001.


## R6 — Interface interativa para validadores e scanner

In [21]:
nome=widgets.Dropdown(options=list(AFDS),description="Formato:")
entrada=widgets.Text(description="Entrada:",placeholder="CH1234")
validar_btn=widgets.Button(description="Validar",button_style="primary")
testar_btn=widgets.Button(description="Testar AFD × RegEx")
out1=widgets.Output()
def validar_ui(_):
    with out1:
        clear_output()
        cfg=AFDS[nome.value]; x=entrada.value
        print("AFD:", "ACEITA" if cfg["afd"].aceita(x) else "REJEITA")
        print("RegEx:", "ACEITA" if re.fullmatch(cfg["regex"],x) else "REJEITA")
def testar_ui(_):
    with out1: clear_output(); testar(nome.value)
validar_btn.on_click(validar_ui); testar_btn.on_click(testar_ui)
ui1=widgets.VBox([nome,entrada,widgets.HBox([validar_btn,testar_btn]),out1])

texto=widgets.Textarea(value="ABRIR CH1234 IMP-001\\nPRIORIDADE CH1234 == 2 @\\nCONSULTAR CH1234 # comentário",
                       description="Comandos:",layout=widgets.Layout(width="100%",height="150px"))
scan_btn=widgets.Button(description="Executar scanner",button_style="primary"); out2=widgets.Output()
def scan_ui(_):
    with out2:
        clear_output(); toks,errs=scanner(texto.value)
        print("TOKENS")
        for t in toks: print(f"{t.tipo:<18} {t.lexema!r:<18} linha={t.linha}, coluna={t.coluna}")
        print("\\nERROS")
        for e in errs: print(f"Linha {e.linha}, coluna {e.coluna}: {e.char!r}. Dica: {e.dica}")
        print(f"\\n{len(toks)} tokens; {len(errs)} erros. O scanner continuou até o fim.")
scan_btn.on_click(scan_ui)
ui2=widgets.VBox([texto,scan_btn,out2])
abas=widgets.Tab(children=[ui1,ui2]); abas.set_title(0,"Validadores AFD × RegEx"); abas.set_title(1,"Mini-scanner")
display(abas)